# COVID-19 Mortality & Risk Factor Analysis — Mexico

### Executive Summary
This notebook processes two epidemiological datasets from the Ministry of Health:

- **Dataset 1 (Recent / Official, `COVID19MEXICO.csv`)**: ~137K records from the current surveillance system, updated to November 18, 2025. It has complete granularity: state of residence, date of symptom onset, date of admission, age, comorbidities, hospitalization, intubation, and death. **It is the only one of the two datasets with state and date**, so it is the one that feeds the Power BI report "Analysis of Covid-19 in Mexico 2025" and its filters for `Date of Symptom Onset`, `Age Group`, and `State`.

- **Dataset 2 (Historical, Kaggle `Covid Data.csv`)**: 1.05M records from the first wave of the pandemic (2020-2021). It lacks columns for residence status and symptom onset date, and its number of rows (1,048,575) falls right at Excel's row limit. It's worth confirming against the original source that it wasn't truncated during export. It is cleaned and stored separately; **it is no longer mixed with the main fact table**, because combining it with Dataset 1 caused some dashboard cards (Hospitalized, Intubated) to silently include these 1.04M records without date or status, while others (Confirmed Cases) remained contained within two distinct universes, responding differently to the same filters.

**Key Objectives:**
1. Clean and standardize both datasets independently.

2. Build a single, fully filterable fact table (`df_recent`, derived from Dataset 1) that matches each filter used in the Power BI report.

3. Export `csv` artifacts — including a state dimension with INEGI/ISO code to feed an actual star schema in Power BI.

In [18]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

def resolve_project_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "data" / "raw" / "COVID19MEXICO.csv").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Mapping variables (SSA / INEGI Official Codes)
STATE_MAPPING = {
    1: "AGUASCALIENTES", 2: "BAJA CALIFORNIA", 3: "BAJA CALIFORNIA SUR",
    4: "CAMPECHE", 5: "COAHUILA DE ZARAGOZA", 6: "COLIMA",
    7: "CHIAPAS", 8: "CHIHUAHUA", 9: "CIUDAD DE MÉXICO",
    10: "DURANGO", 11: "GUANAJUATO", 12: "GUERRERO",
    13: "HIDALGO", 14: "JALISCO", 15: "MÉXICO",
    16: "MICHOACÁN DE OCAMPO", 17: "MORELOS", 18: "NAYARIT",
    19: "NUEVO LEÓN", 20: "OAXACA", 21: "PUEBLA",
    22: "QUERÉTARO", 23: "QUINTANA ROO", 24: "SAN LUIS POTOSÍ",
    25: "SINALOA", 26: "SONORA", 27: "TABASCO",
    28: "TAMAULIPAS", 29: "TLAXCALA", 30: "VERACRUZ DE IGNACIO DE LA LLAVE",
    31: "YUCATÁN", 32: "ZACATECAS", 99: "NOT_SPECIFIED",
}

INEGI_TO_ISO_MAPPING = {
    f"{i:02d}": f"MX{code}" for i, code in zip(range(1, 33), [
        "AGS", "BCN", "BCS", "CAM", "COA", "COL", "CHP", "CHH", "MEX", "DUR",
        "GUA", "GRO", "HID", "JAL", "MEX", "MIC", "MOR", "NAY", "NLE", "OAX",
        "PUE", "QUE", "ROO", "SLP", "SIN", "SON", "TAB", "TAM", "TLA", "VER",
        "YUC", "ZAC"
    ])
}

# Column Translation Dictionary 
COLUMN_TRANSLATION = {
    "USMER": "ORIGIN", "MEDICAL_UNIT": "SECTOR", "SEX": "SEX",
    "PATIENT_TYPE": "PATIENT_TYPE", "DATE_DIED": "DEATH_DATE",
    "INTUBED": "INTUBATED", "PNEUMONIA": "PNEUMONIA", "AGE": "AGE",
    "PREGNANT": "PREGNANT", "DIABETES": "DIABETES", "COPD": "COPD",
    "ASTHMA": "ASTHMA", "INMSUPR": "IMMUNOSUPPRESSED", "HIPERTENSION": "HYPERTENSION",
    "OTHER_DISEASE": "OTHER_COMORBIDITY", "CARDIOVASCULAR": "CARDIOVASCULAR",
    "OBESITY": "OBESITY", "RENAL_CHRONIC": "CHRONIC_RENAL",
    "TOBACCO": "TOBACCO_USE", "ICU": "ICU",
    "CLASIFICATION_FINAL": "FINAL_COVID_CLASSIFICATION",  
    "CLASIFFICATION_FINAL": "FINAL_COVID_CLASSIFICATION",  
}

# Translation dictionary for the Spanish Dataset (df1)
SPANISH_COLUMN_TRANSLATION = {
    "FECHA_ACTUALIZACION": "LAST_UPDATED",
    "ID_REGISTRO": "REGISTRATION_ID",
    "ORIGEN": "ORIGIN",
    "SECTOR": "SECTOR",
    "ENTIDAD_UM": "ENTIDAD_UM",           
    "SEXO": "SEX",
    "ENTIDAD_NAC": "ENTIDAD_NAC",       
    "ENTIDAD_RES": "ENTIDAD_RES",         
    "MUNICIPIO_RES": "MUNICIPIO_RES",     
    "TIPO_PACIENTE": "PATIENT_TYPE",
    "FECHA_INGRESO": "ADMISSION_DATE",
    "FECHA_SINTOMAS": "SYMPTOMS_ONSET_DATE",
    "FECHA_DEF": "DEATH_DATE",
    "INTUBADO": "INTUBATED",
    "NEUMONIA": "PNEUMONIA",
    "EDAD": "AGE",
    "NACIONALIDAD": "NATIONALITY",
    "EMBARAZO": "PREGNANT",
    "HABLA_LENGUA_INDIG": "SPEAKS_INDIGENOUS_LANGUAGE",
    "INDIGENA": "INDIGENOUS",
    "DIABETES": "DIABETES",
    "EPOC": "COPD",
    "ASMA": "ASTHMA",
    "INMUSUPR": "IMMUNOSUPPRESSED",
    "HIPERTENSION": "HYPERTENSION",
    "OTRA_COM": "OTHER_COMORBIDITY",
    "CARDIOVASCULAR": "CARDIOVASCULAR",
    "OBESIDAD": "OBESITY", 
    "RENAL_CRONICA": "CHRONIC_RENAL",
    "TABAQUISMO": "TOBACCO_USE",
    "OTRO_CASO": "CONTACT_WITH_OTHER_CASE",
    "TOMA_MUESTRA_LAB": "LAB_SAMPLE_TAKEN",
    "RESULTADO_PCR": "PCR_RESULT",
    "RESULTADO_PCR_COINFECCION": "PCR_COINFECTION_RESULT",
    "TOMA_MUESTRA_ANTIGENO": "ANTIGEN_SAMPLE_TAKEN",
    "RESULTADO_ANTIGENO": "ANTIGEN_RESULT",
    "CLASIFICACION_FINAL_COVID": "FINAL_COVID_CLASSIFICATION",
    "CLASIFICACION_FINAL_FLU": "FINAL_FLU_CLASSIFICATION",
    "MIGRANTE": "MIGRANT",
    "PAIS_NACIONALIDAD": "COUNTRY_OF_NATIONALITY",
    "PAIS_ORIGEN": "COUNTRY_OF_ORIGIN",
    "UCI": "ICU"
}

BINARY_COLUMNS = [
    "DIABETES", "COPD", "ASTHMA", "IMMUNOSUPPRESSED", "HYPERTENSION", "OTHER_COMORBIDITY",
    "CARDIOVASCULAR", "OBESITY", "CHRONIC_RENAL", "TOBACCO_USE",
    "INTUBATED", "PNEUMONIA", "ICU", "PREGNANT",
]

COMORBIDITIES_FOR_ANALYSIS = [
    "DIABETES", "HYPERTENSION", "OBESITY", "COPD", "ASTHMA",
    "CHRONIC_RENAL", "CARDIOVASCULAR", "TOBACCO_USE", "IMMUNOSUPPRESSED", "OTHER_COMORBIDITY",
]

def export_processed_csv(df: pd.DataFrame, file_name: str) -> None:
    output_path = PROCESSED_DIR / file_name
    df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Exported artifact: {output_path}")

print("Configuration loaded.")
print(f"  Project root : {PROJECT_ROOT}")
print(f"  Raw data dir : {RAW_DIR}")
print(f"  Output dir   : {PROCESSED_DIR}")
print(f"  COVID19 CSV  : {(RAW_DIR / 'COVID19MEXICO.csv').exists()}")
print(f"  Covid Data   : {(RAW_DIR / 'Covid Data.csv').exists()}")

Configuration loaded.
  Project root : c:\Users\willi\Documents\proyectos\covid19-mortality-analysis
  Raw data dir : c:\Users\willi\Documents\proyectos\covid19-mortality-analysis\data\raw
  Output dir   : c:\Users\willi\Documents\proyectos\covid19-mortality-analysis\data\processed
  COVID19 CSV  : True
  Covid Data   : True


# 1. Data Loading & Memory Optimization

In [19]:
# 1. Data Loading & Memory Optimization
optimal_dtypes = {
    'SECTOR': 'category',
    'ENTIDAD_UM': 'category',
    'ENTIDAD_NAC': 'category',
    'ENTIDAD_RES': 'category',
    'MUNICIPIO_RES': 'category',
    'TIPO_PACIENTE': 'category',
    'NACIONALIDAD': 'category'
}

print("Ingesting datasets...")
df1 = pd.read_csv(RAW_DIR / "COVID19MEXICO.csv", dtype=optimal_dtypes, low_memory=False)
df2 = pd.read_csv(RAW_DIR / "Covid Data.csv", low_memory=False)

print(f"Dataset 1 (Recent) loaded: {len(df1):,} rows")
print(f"Dataset 2 (Historical) loaded: {len(df2):,} rows")

Ingesting datasets...
Dataset 1 (Recent) loaded: 137,030 rows
Dataset 2 (Historical) loaded: 1,048,575 rows


In [20]:
df2.head()

,USMER,MEDICAL_UNIT,SEX,PATIENT_TYPE,DATE_DIED,INTUBED,PNEUMONIA,AGE,PREGNANT,DIABETES,...,ASTHMA,INMSUPR,HIPERTENSION,OTHER_DISEASE,CARDIOVASCULAR,OBESITY,RENAL_CHRONIC,TOBACCO,CLASIFFICATION_FINAL,ICU
0,2,1,1,1,03/05/2020,97,1,65,2,2,...,2,2,1,2,2,2,2,2,3,97
1,2,1,2,1,03/06/2020,97,1,72,97,2,...,2,2,1,2,2,1,1,2,5,97
2,2,1,2,2,09/06/2020,1,2,55,97,1,...,2,2,2,2,2,2,2,2,3,2
3,2,1,1,1,12/06/2020,97,2,53,2,2,...,2,2,2,2,2,2,2,2,7,97
4,2,1,2,1,21/06/2020,97,2,68,97,1,...,2,2,1,2,2,2,2,2,3,97


## 2. Harmonization, Cleaning, and Unification

In [21]:
# Rename df1 using the Spanish-to-English map
df1 = df1.rename(columns=SPANISH_COLUMN_TRANSLATION)

# Rename df2 using your original English-to-Clean-English map (COLUMN_TRANSLATION)
df2 = df2.rename(columns=COLUMN_TRANSLATION)

print("Columns successfully standardized. Ready for merging")

Columns successfully standardized. Ready for merging


### 2.1 Date parsing by source, before unifying

`df1` (official) stores its dates in ISO `YYYY-MM-DD` format. `df2` (historical, Kaggle) stores them in `DD/MM/YYYY`, and uses `"9999-99-99"` as a sentinel value for "did not die". Previously, `DEATH_DATE` was parsed *after* concatenating both datasets with `pd.to_datetime(..., errors="coerce")` without specifying the format, causing pandas to guess a single format for the entire concatenated column. This caused two problems: ambiguous dates like `03/05/2020` were interpreted as March 5th instead of May 3rd, and unambiguous dates like `25/06/2020` were discarded as null.


In [22]:
for col in ["ADMISSION_DATE", "SYMPTOMS_ONSET_DATE", "DEATH_DATE"]:
    if col in df1.columns:
        df1[col] = pd.to_datetime(df1[col], format="%Y-%m-%d", errors="coerce")

if "DEATH_DATE" in df2.columns:
    df2["DEATH_DATE"] = pd.to_datetime(df2["DEATH_DATE"], format="%d/%m/%Y", errors="coerce")

print("Dates parsed by source, before merging:")
print(f"  df1 DEATH_DATE no nulo: {df1['DEATH_DATE'].notna().sum():,}")
print(f"  df2 DEATH_DATE no nulo: {df2['DEATH_DATE'].notna().sum():,}")

Dates parsed by source, before merging:
  df1 DEATH_DATE no nulo: 5,090
  df2 DEATH_DATE no nulo: 76,942


In [23]:
print(f"Codes in df1 for DIABETES: {df1['DIABETES'].unique()}")
print(f"Codes in df2 for DIABETES: {df2['DIABETES'].unique()}")

SPECIAL_MISSING_CODES = {97: np.nan, 98: np.nan, 99: np.nan}

for df in (df1, df2):
    for col in BINARY_COLUMNS:
        if col in df.columns:
            df[col] = df[col].replace(SPECIAL_MISSING_CODES)
            df[col] = pd.to_numeric(df[col], errors="coerce")

print("Cleanup of code 97/98/99 (no aplica / no sabe / no especificado) completed.")

Codes in df1 for DIABETES: [ 2  1 98]
Codes in df2 for DIABETES: [ 2  1 98]
Cleanup of code 97/98/99 (no aplica / no sabe / no especificado) completed.


### 2.2 Scope decision: why `df1` and `df2` are no longer unified

`df2` (historical, Kaggle) does not have a residence status or symptom onset date, so merging it with `df1` broke the consistency of the Date/Status/Age filters on the dashboard between KPI cards. `df1`, the recent dataset, current as of 2025-11-18, becomes the only fact table (`df_recent`); `df2` is cleaned and exported separately (`df_historical`).

In [24]:
df_recent = df1.copy()
df_historical = df2.copy()

print(f"df_recent (fact table del dashboard 2025): {len(df_recent):,} filas, {df_recent.shape[1]} columnas")
print(f"df_historical (Kaggle 2020-2021, separado):  {len(df_historical):,} filas, {df_historical.shape[1]} columnas")
df_recent.head()

df_recent["PATIENT_TYPE"] = pd.to_numeric(df_recent["PATIENT_TYPE"], errors="coerce")
df_historical["PATIENT_TYPE"] = pd.to_numeric(df_historical["PATIENT_TYPE"], errors="coerce")

print(f"PATIENT_TYPE corregido a numerico. Hospitalizados reales en df_recent: {(df_recent['PATIENT_TYPE'] == 2).sum():,}")

df_recent (fact table del dashboard 2025): 137,030 filas, 42 columnas
df_historical (Kaggle 2020-2021, separado):  1,048,575 filas, 21 columnas
PATIENT_TYPE corregido a numerico. Hospitalizados reales en df_recent: 62,664


## 3. Feature Engineering (Derived Variables)

In [25]:
df_recent["DEATH_DATE"] = pd.to_datetime(df_recent["DEATH_DATE"], errors="coerce")
df_recent["DECEASED"] = df_recent["DEATH_DATE"].notna().astype(int)

df_historical["DEATH_DATE"] = pd.to_datetime(df_historical["DEATH_DATE"], errors="coerce")
df_historical["DECEASED"] = df_historical["DEATH_DATE"].notna().astype(int)

print("Death count analysis (df_recent — dataset del dashboard 2025):")
print(df_recent["DECEASED"].value_counts())

Death count analysis (df_recent — dataset del dashboard 2025):
DECEASED
0    131940
1      5090
Name: count, dtype: int64


In [26]:
df_recent["AGE"] = pd.to_numeric(df_recent["AGE"], errors="coerce")

age_bins = [0, 18, 36, 60, df_recent["AGE"].max() + 1]
age_labels = ["0-17", "18-35", "36-59", "60+"]

df_recent["AGE_GROUP"] = pd.cut(
    df_recent["AGE"],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True,
)

print("Value counts by age group (df_recent):")
print(df_recent["AGE_GROUP"].value_counts(dropna=False))

Value counts by age group (df_recent):
AGE_GROUP
0-17     40776
36-59    35767
18-35    31147
60+      29340
Name: count, dtype: int64


## 4. Case Fatality Rate (CFR) Metrics

### 4.1 CFR by Federal Entity (State)

In [27]:
cfr_by_state = (
    df_recent.groupby("ENTIDAD_RES")["DECEASED"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)
cfr_by_state.columns = ["ENTIDAD_RES", "CFR (%)"]

cfr_by_state["State_Code"] = cfr_by_state["ENTIDAD_RES"].astype(str).str.zfill(2)
cfr_by_state["State"] = pd.to_numeric(cfr_by_state["ENTIDAD_RES"], errors="coerce").map(STATE_MAPPING)

cfr_by_state["ISO_Code"] = cfr_by_state["State_Code"].map(INEGI_TO_ISO_MAPPING)

cfr_by_state = cfr_by_state[["State_Code", "State", "ISO_Code", "CFR (%)"]]

print("Case Fatality Rate (CFR) by State:")
print(cfr_by_state.round(2))

export_processed_csv(cfr_by_state, "cfr_by_state_final_for_map.csv")

dim_estado = pd.DataFrame({"State_Code": [f"{k:02d}" for k in STATE_MAPPING if k != 99]})
dim_estado["State"] = pd.to_numeric(dim_estado["State_Code"]).map(STATE_MAPPING)
dim_estado["ISO_Code"] = dim_estado["State_Code"].map(INEGI_TO_ISO_MAPPING)

print("\nDimensión de estado (para relacionar con la fact table en Power BI):")
print(dim_estado)

export_processed_csv(dim_estado, "dim_estado.csv")

Case Fatality Rate (CFR) by State:
   State_Code                            State ISO_Code  CFR (%)
0          12                         GUERRERO    MXGRO     9.04
1          02                  BAJA CALIFORNIA    MXBCN     8.69
2          23                     QUINTANA ROO    MXROO     8.46
3          28                       TAMAULIPAS    MXTAM     8.01
4          16              MICHOACÁN DE OCAMPO    MXMIC     7.63
5          17                          MORELOS    MXMOR     6.54
6          19                       NUEVO LEÓN    MXNLE     6.42
7          06                           COLIMA    MXCOL     6.34
8          24                  SAN LUIS POTOSÍ    MXSLP     5.77
9          25                          SINALOA    MXSIN     5.59
10         08                        CHIHUAHUA    MXCHH     5.42
11         20                           OAXACA    MXOAX     5.25
12         18                          NAYARIT    MXNAY     5.24
13         14                          JALISCO    MXJAL

### 4.2 CFR by Age Cohort

In [28]:
cfr_by_age_group = (
    df_recent.groupby("AGE_GROUP")["DECEASED"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)
cfr_by_age_group.columns = ["AGE_GROUP", "CFR (%)"]

print("Case Fatality Rate (CFR) by Age Group:")
print(cfr_by_age_group.round(2))

export_processed_csv(cfr_by_age_group, "cfr_by_age_group.csv")

Case Fatality Rate (CFR) by Age Group:
  AGE_GROUP  CFR (%)
0       60+    10.31
1     36-59     3.37
2     18-35     1.26
3      0-17     1.14
Exported artifact: c:\Users\willi\Documents\proyectos\covid19-mortality-analysis\data\processed\cfr_by_age_group.csv


### 4.3 Specific CFR by Comorbidity

In [29]:
fatality_results = {}

for comorbidity in COMORBIDITIES_FOR_ANALYSIS:
    df_filtered = df_recent[df_recent[comorbidity] == 1]
    fatality_results[comorbidity] = (
        df_filtered["DECEASED"].mean() * 100 if not df_filtered.empty else np.nan
    )

df_top_5 = (
    pd.DataFrame(fatality_results.items(), columns=["Comorbidity", "Specific CFR (%)"])
    .sort_values(by="Specific CFR (%)", ascending=False)
)

print("Top 5 comorbidities with the highest specific CFR (df_recent):")
print(df_top_5.head(5).round(2))

export_processed_csv(df_top_5, "top5_comorbidities.csv")

Top 5 comorbidities with the highest specific CFR (df_recent):
      Comorbidity  Specific CFR (%)
5   CHRONIC_RENAL             12.46
6  CARDIOVASCULAR             12.19
3            COPD             11.75
0        DIABETES              9.73
1    HYPERTENSION              9.15
Exported artifact: c:\Users\willi\Documents\proyectos\covid19-mortality-analysis\data\processed\top5_comorbidities.csv


### 4.4 Intubation Proportion among Hospitalized Patients

In [30]:
df_hospitalized = df_recent[df_recent["PATIENT_TYPE"] == 2].copy()
total_hospitalized = len(df_hospitalized)
total_intubated = (df_hospitalized["INTUBATED"] == 1).sum()
intubation_ratio = (
    (total_intubated / total_hospitalized) * 100 if total_hospitalized > 0 else 0
)

df_intubation_analysis = pd.DataFrame(
    {
        "Metric": [
            "Total Hospitalized Patients",
            "Intubated Patients",
            "Intubation Ratio (%)",
        ],
        "Value": [total_hospitalized, total_intubated, intubation_ratio],
    }
)

print("Intubation analysis for hospitalized patients (df_recent):")
print(df_intubation_analysis)

export_processed_csv(df_intubation_analysis, "intubation_comparison.csv")

Intubation analysis for hospitalized patients (df_recent):
                        Metric         Value
0  Total Hospitalized Patients  62664.000000
1           Intubated Patients   2936.000000
2         Intubation Ratio (%)      4.685306
Exported artifact: c:\Users\willi\Documents\proyectos\covid19-mortality-analysis\data\processed\intubation_comparison.csv


### 4.5 Time Series: Confirmed Cases, Deaths, and Daily CFR

In [31]:
df_recent["SYMPTOMS_ONSET_DATE"] = pd.to_datetime(
    df_recent["SYMPTOMS_ONSET_DATE"], errors="coerce"
)
df_recent["DEATH_DATE"] = pd.to_datetime(df_recent["DEATH_DATE"], errors="coerce")

confirmed_cases = df_recent[
    df_recent["FINAL_COVID_CLASSIFICATION"].isin([1, 2, 3])
    & df_recent["SYMPTOMS_ONSET_DATE"].notna()
].copy()

cases_series = (
    confirmed_cases.groupby("SYMPTOMS_ONSET_DATE")
    .size()
    .reset_index(name="Confirmed_Cases")
    .rename(columns={"SYMPTOMS_ONSET_DATE": "Date"})
)

deaths_df = confirmed_cases[confirmed_cases["DEATH_DATE"].notna()].copy()
deaths_series = (
    deaths_df.groupby("DEATH_DATE")
    .size()
    .reset_index(name="Deaths")
    .rename(columns={"DEATH_DATE": "Date"})
)

final_time_series = (
    pd.merge(cases_series, deaths_series, on="Date", how="outer")
    .sort_values("Date")
    .fillna(0)
)
final_time_series["Confirmed_Cases"] = final_time_series["Confirmed_Cases"].astype(int)
final_time_series["Deaths"] = final_time_series["Deaths"].astype(int)
final_time_series["Daily_CFR (%)"] = np.where(
    final_time_series["Confirmed_Cases"] > 0,
    (final_time_series["Deaths"] / final_time_series["Confirmed_Cases"]) * 100,
    0.0,
)

print("Time series preview (df_recent):")
print(final_time_series.tail().round(2))

export_processed_csv(final_time_series, "cases_deaths_time_series.csv")

Time series preview (df_recent):
          Date  Confirmed_Cases  Deaths  Daily_CFR (%)
312 2025-11-09                5       0            0.0
313 2025-11-10                3       0            0.0
314 2025-11-11                4       0            0.0
315 2025-11-12                2       0            0.0
316 2025-11-13                3       0            0.0
Exported artifact: c:\Users\willi\Documents\proyectos\covid19-mortality-analysis\data\processed\cases_deaths_time_series.csv


### 4.6 CFR by Cumulative Comorbidity Count

In [32]:
df_count = df_recent[["DECEASED"] + COMORBIDITIES_FOR_ANALYSIS].copy()
df_count["NUM_COMORBIDITIES"] = df_count[COMORBIDITIES_FOR_ANALYSIS].eq(1).sum(axis=1)
df_recent["NUM_COMORBIDITIES"] = df_count["NUM_COMORBIDITIES"]

comorb_bins = [-0.5, 0.5, 1.5, 2.5, df_recent["NUM_COMORBIDITIES"].max() + 1]
comorb_labels = ["0 comorbidities", "1 comorbidity", "2 comorbidities", "3+ comorbidities"]

df_recent["COMORBIDITY_GROUP"] = pd.cut(
    df_recent["NUM_COMORBIDITIES"],
    bins=comorb_bins,
    labels=comorb_labels,
    right=False,
    include_lowest=True,
)

cfr_by_comorbidity_count = (
    df_recent.groupby("COMORBIDITY_GROUP")["DECEASED"]
    .mean()
    .mul(100)
    .reset_index(name="CFR (%)")
)
cfr_by_comorbidity_count.columns = ["Comorbidity Group", "CFR (%)"]

print("Case Fatality Rate (CFR) by Number of Comorbidities (df_recent):")
print(cfr_by_comorbidity_count.round(2))

export_processed_csv(cfr_by_comorbidity_count, "deaths_by_comorbidity_count.csv")

Case Fatality Rate (CFR) by Number of Comorbidities (df_recent):
  Comorbidity Group  CFR (%)
0   0 comorbidities     1.74
1     1 comorbidity     5.07
2   2 comorbidities     8.63
3  3+ comorbidities    11.33
Exported artifact: c:\Users\willi\Documents\proyectos\covid19-mortality-analysis\data\processed\deaths_by_comorbidity_count.csv


## 5. Master Dataset Export

In [33]:
FACT_TABLE_COLUMNS = [
    "REGISTRATION_ID", "ENTIDAD_RES", "SEX", "PATIENT_TYPE", "AGE", "AGE_GROUP",
    "ADMISSION_DATE", "SYMPTOMS_ONSET_DATE", "DEATH_DATE", "DECEASED",
    "INTUBATED", "PNEUMONIA", "ICU", "PREGNANT",
    *COMORBIDITIES_FOR_ANALYSIS, "NUM_COMORBIDITIES", "COMORBIDITY_GROUP",
    "FINAL_COVID_CLASSIFICATION",
]
export_processed_csv(df_recent[FACT_TABLE_COLUMNS], "fact_covid_2025.csv")

export_processed_csv(df_historical, "historical_2020_2021_clean.csv")

Exported artifact: c:\Users\willi\Documents\proyectos\covid19-mortality-analysis\data\processed\fact_covid_2025.csv
Exported artifact: c:\Users\willi\Documents\proyectos\covid19-mortality-analysis\data\processed\historical_2020_2021_clean.csv


# 6. Final Dataset Validation

In [34]:
print("DataFrame Preview (df_recent — fact table del dashboard):")
print(df_recent[FACT_TABLE_COLUMNS].head())

print("\nDataFrame Structure and Memory Usage:")
print(df_recent[FACT_TABLE_COLUMNS].info())

print("\nComplete List of Fact Table Columns:")
for col in FACT_TABLE_COLUMNS:
    print(f"'{col}',")

DataFrame Preview (df_recent — fact table del dashboard):
  REGISTRATION_ID ENTIDAD_RES  SEX  PATIENT_TYPE  AGE AGE_GROUP  \
0         g771db5          21    1             1   15      0-17   
1         g574d0a          09    1             1   30     18-35   
2         g941a21          10    1             1   34     18-35   
3         geaaed1          09    2             2   87       60+   
4         g9c2f41          24    2             2    2      0-17   

  ADMISSION_DATE SYMPTOMS_ONSET_DATE DEATH_DATE  DECEASED  ...  COPD  ASTHMA  \
0     2025-02-10          2025-02-09        NaT         0  ...   2.0     2.0   
1     2025-02-10          2025-02-10        NaT         0  ...   2.0     2.0   
2     2025-02-11          2025-02-11        NaT         0  ...   2.0     2.0   
3     2025-02-10          2025-02-10        NaT         0  ...   2.0     2.0   
4     2025-02-10          2025-02-10        NaT         0  ...   2.0     2.0   

   CHRONIC_RENAL  CARDIOVASCULAR  TOBACCO_USE  IMMUNOSUPPR